In [0]:
%sql
CREATE OR REPLACE TEMP VIEW products_typed AS
SELECT
  product_id,
  product_category_name,
  TRY_CAST(product_name_lenght AS INT)        AS product_name_length,
  TRY_CAST(product_description_lenght AS INT) AS product_description_length,
  TRY_CAST(product_photos_qty AS INT)         AS product_photos_qty,
  TRY_CAST(product_weight_g AS INT)           AS product_weight_g,
  TRY_CAST(product_length_cm AS INT)          AS product_length_cm,
  TRY_CAST(product_height_cm AS INT)          AS product_height_cm,
  TRY_CAST(product_width_cm AS INT)           AS product_width_cm,
  _ingested_at
FROM workspace.bronze.products
QUALIFY ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY _ingested_at DESC) = 1;

CREATE OR REPLACE TEMP VIEW products_validated AS
SELECT *,
  CASE
    WHEN product_id IS NULL THEN 'llave nula'
    WHEN product_weight_g < 0 OR product_length_cm < 0
      OR product_height_cm < 0 OR product_width_cm < 0 THEN 'medidas negativas'
  END AS motivo
FROM products_typed;

CREATE OR REPLACE TABLE workspace.silver.products AS
SELECT * EXCEPT (motivo) FROM products_validated WHERE motivo IS NULL;

CREATE OR REPLACE TABLE workspace.silver.products_quarantine AS
SELECT * FROM products_validated WHERE motivo IS NOT NULL;